<a href="https://colab.research.google.com/github/adhlaphy/enose-pca-fingerprinting/blob/main/01_PCA_Drift_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install ucimlrepo --quiet
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from ucimlrepo import fetch_ucirepo
import os, zipfile
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
%matplotlib inline

print("All libraries loaded successfully")
print(f"Numpy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")

In [ ]:
gas_names = {
    1: 'Ethanol',
    2: 'Ethylene',
    3: 'Ammonia',
    4: 'Acetaldehyde',
    5: 'Acetone',
    6: 'Toluene'
}

gas_colors = {
    'Ethanol':      '#E63946',
    'Ethylene':     '#2196F3',
    'Ammonia':      '#4CAF50',
    'Acetaldehyde': '#FF9800',
    'Acetone':      '#9C27B0',
    'Toluene':      '#00BCD4'
}
BASE    = '/content/drive/MyDrive/PhD_projects/Project1_Enose_PCA'
RAW_DIR = BASE + '/Data/raw/'
FIG_DIR = BASE + '/Figures/'

os.makedirs(FIG_DIR, exist_ok=True)

plt.rcParams.update({
    'figure.dpi'        : 150,
    'font.family'       : 'DejaVu Serif',
    'font.size'         : 11,
    'axes.labelsize'    : 12,
    'axes.titlesize'    : 13,
    'legend.fontsize'   : 10,
    'axes.spines.top'   : False,
    'axes.spines.right' : False,
    'axes.linewidth'    : 1.2,
})

print("✓ gas_names:", list(gas_names.values()))
print("✓ Paths set")
print(f"  BASE    = {BASE}")
print(f"  RAW_DIR = {RAW_DIR}")
print(f"  FIG_DIR = {FIG_DIR}")

In [ ]:
zip_path = BASE + '/Data/gas+sensor+array+drift+dataset.zip'

if not os.path.exists(RAW_DIR) or len(os.listdir(RAW_DIR)) == 0:
    os.makedirs(RAW_DIR, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(RAW_DIR)
    print("✓ Unzipped successfully")
else:
    print("✓ Already unzipped, skipping")

print("\nContents of raw folder:")
for f in sorted(os.listdir(RAW_DIR)):
    print(f"  {f}")

In [ ]:
def parse_dat_file(filepath):
    """
    Each line in the .dat file looks like:
      3 1:0.342 2:-0.11 3:0.985 ... 128:0.456
      │ └─────────────────────────────────────── 128 features
      └── gas label (1 to 6)
    """
    X_list, y_list = [], []
    with open(filepath, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split()
            label    = int(parts[0])
            features = np.zeros(128)
            for item in parts[1:]:
                idx, val = item.split(':')
                features[int(idx) - 1] = float(val)
            X_list.append(features)
            y_list.append(label)
    return np.array(X_list), np.array(y_list)


# Find all .dat files (searches subfolders too)
dat_files = []
for root, dirs, files in os.walk(RAW_DIR):
    for file in sorted(files):
        if file.endswith('.dat'):
            dat_files.append(os.path.join(root, file))

print(f"Found {len(dat_files)} .dat files\n")

# Load all batches
X_all, y_all, batch_ids = [], [], []
for batch_num, filepath in enumerate(dat_files, start=1):
    X_batch, y_batch = parse_dat_file(filepath)
    X_all.append(X_batch)
    y_all.append(y_batch)
    batch_ids.extend([batch_num] * len(y_batch))
    print(f"  Batch {batch_num:2d}: {X_batch.shape[0]:4d} samples")

X         = np.vstack(X_all)
y         = np.concatenate(y_all)
batch_ids = np.array(batch_ids)

print(f"\n✓ X shape : {X.shape}")
print(f"✓ y shape : {y.shape}")
print(f"✓ Batches : {np.unique(batch_ids)}")
print(f"✓ Gas labels: {np.unique(y)}")

print("\n── Samples per gas ──")
for num, name in gas_names.items():
    print(f"  {name:>15}: {np.sum(y==num):>5}")

In [ ]:
scaler       = StandardScaler()
X_normalized = scaler.fit_transform(X)

# Verify
print("── Before normalization ──")
print(f"  Feature 1 → mean: {X[:,0].mean():8.4f}  std: {X[:,0].std():8.4f}")
print(f"  Feature 2 → mean: {X[:,1].mean():8.4f}  std: {X[:,1].std():8.4f}")

print("\n── After normalization ──")
print(f"  Feature 1 → mean: {X_normalized[:,0].mean():8.4f}  std: {X_normalized[:,0].std():8.4f}")
print(f"  Feature 2 → mean: {X_normalized[:,1].mean():8.4f}  std: {X_normalized[:,1].std():8.4f}")

print(f"\n✓ X_normalized shape: {X_normalized.shape}")
print("\nSetup complete. All variables ready:")
print("  X            — raw data         (13910, 128)")
print("  X_normalized — normalized data  (13910, 128)")
print("  y            — gas labels       (13910,)")
print("  batch_ids    — batch numbers    (13910,)")
print("  gas_names    — label dictionary")
print("  gas_colors   — color dictionary")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

feature_indices = [0, 1, 8, 15]
feature_labels  = [
    'Feature 1  (Sensor 1 Steady-State)',
    'Feature 2  (Sensor 1 Transient)',
    'Feature 9  (Sensor 5 Steady-State)',
    'Feature 16 (Sensor 8 Steady-State)'
]

for ax, feat_idx, feat_label in zip(axes.ravel(), feature_indices, feature_labels):
    for gas_num, gas_name in gas_names.items():
        mask   = (y == gas_num)
        values = X[mask, feat_idx]
        ax.hist(values,
                bins=50, alpha=0.45,
                color=gas_colors[gas_name],
                label=gas_name,
                density=True)
    ax.set_xlabel(feat_label, fontsize=10)
    ax.set_ylabel('Probability Density', fontsize=10)
    ax.set_title(feat_label, fontsize=10)

# Shared legend at bottom
handles, labels = axes[0,0].get_legend_handles_labels()
fig.legend(handles, labels,
           loc='lower center', ncol=6,
           frameon=False, fontsize=10,
           bbox_to_anchor=(0.5, -0.02))

fig.suptitle('Single Features Cannot Separate All 6 Gases — This Motivates PCA',
             fontsize=12, fontweight='bold', y=1.01)

plt.tight_layout()
plt.savefig(FIG_DIR + 'feature_distributions.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Saved")

In [ ]:
X_first16    = X[:, :16]
corr_matrix  = np.corrcoef(X_first16.T)

# Find most correlated pair
mask = ~np.eye(16, dtype=bool)          # Ignore diagonal
flat_idx  = np.abs(corr_matrix * mask).argmax()
row, col  = np.unravel_index(flat_idx, (16, 16))
print(f"Most correlated pair: S{row+1} and S{col+1}")
print(f"Their correlation   : {corr_matrix[row, col]:.4f}")

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_matrix,
            ax=ax,
            cmap='coolwarm', center=0, vmin=-1, vmax=1,
            annot=True, fmt='.2f', annot_kws={'size': 7},
            square=True, linewidths=0.4, linecolor='white',
            cbar_kws={'label': 'Pearson Correlation', 'shrink': 0.85})

labels = [f'S{i+1}' for i in range(16)]
ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(labels, rotation=0, fontsize=9)
ax.set_title('Feature Correlation Matrix — High Correlation Motivates PCA',
             fontsize=12, pad=15)

plt.tight_layout()
plt.savefig(FIG_DIR + 'correlation_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Saved")

In [ ]:
def pca_from_scratch(X, n_components=10):
    N, p = X.shape

    # Step 1: Center
    mean_vec  = np.mean(X, axis=0)
    X_c       = X - mean_vec

    # Step 2: Covariance matrix
    C = (X_c.T @ X_c) / (N - 1)

    # Step 3: Eigendecomposition
    eigenvalues, eigenvectors = np.linalg.eigh(C)

    # Step 4: Sort descending
    order        = np.argsort(eigenvalues)[::-1]
    eigenvalues  = eigenvalues[order]
    eigenvectors = eigenvectors[:, order]

    # Step 5: Project
    Z   = X_c @ eigenvectors[:, :n_components]
    evr = eigenvalues / eigenvalues.sum()

    return Z, eigenvectors, eigenvalues, evr


Z_manual, eigvecs, eigvals, evr = pca_from_scratch(X_normalized, n_components=10)

print("── Explained Variance per Component ──")
print(f"{'PC':<6} {'Individual':>12} {'Cumulative':>12}")
print("─" * 32)
cumulative = 0
for i in range(10):
    cumulative += evr[i]
    print(f"PC{i+1:<4}  {evr[i]*100:>11.2f}%  {cumulative*100:>11.2f}%")

print(f"\n✓ Z_manual shape: {Z_manual.shape}")

In [ ]:
pca_sk  = PCA(n_components=10)
Z_sk    = pca_sk.fit_transform(X_normalized)

print(f"{'PC':<6} {'Manual %':>10} {'sklearn %':>10} {'Match':>8}")
print("─" * 38)
for i in range(10):
    m  = evr[i] * 100
    s  = pca_sk.explained_variance_ratio_[i] * 100
    ok = "✓" if abs(m - s) < 0.01 else "✗"
    print(f"PC{i+1:<4}  {m:>10.4f}  {s:>10.4f}  {ok:>8}")

print("\n── Projection correlation (should be ~1.0) ──")
for i in range(3):
    r1 = abs(np.corrcoef(Z_manual[:,i],  Z_sk[:,i])[0,1])
    r2 = abs(np.corrcoef(Z_manual[:,i], -Z_sk[:,i])[0,1])
    print(f"  PC{i+1}: {max(r1,r2):.8f}")

In [ ]:
pca_full = PCA(n_components=128)
pca_full.fit(X_normalized)

evr_full  = pca_full.explained_variance_ratio_          # All 128 values
evr_cumul = np.cumsum(evr_full)                         # Running total

# ── Find key thresholds ────────────────────────────────────────
n_80 = np.argmax(evr_cumul >= 0.80) + 1   # PCs needed for 80%
n_90 = np.argmax(evr_cumul >= 0.90) + 1   # PCs needed for 90%
n_95 = np.argmax(evr_cumul >= 0.95) + 1   # PCs needed for 95%

print(f"PCs needed for 80% variance: {n_80}")
print(f"PCs needed for 90% variance: {n_90}")
print(f"PCs needed for 95% variance: {n_95}")
print(f"\nPC1 alone : {evr_full[0]*100:.2f}%")
print(f"PC2 alone : {evr_full[1]*100:.2f}%")
print(f"PC1+PC2   : {evr_cumul[1]*100:.2f}%")

# ── Plot ───────────────────────────────────────────────────────
fig, ax1 = plt.subplots(figsize=(10, 5))

# Bar chart — individual explained variance (left axis)
x = np.arange(1, 21)               # Show first 20 PCs
ax1.bar(x, evr_full[:20] * 100,
        color='#2196F3', alpha=0.7,
        edgecolor='white', linewidth=0.5,
        label='Individual EVR')

ax1.set_xlabel('Principal Component Number', fontsize=12)
ax1.set_ylabel('Explained Variance (%)', color='#2196F3', fontsize=12)
ax1.tick_params(axis='y', labelcolor='#2196F3')
ax1.set_xticks(x)
ax1.set_xlim(0.5, 20.5)

# Line chart — cumulative explained variance (right axis)
ax2 = ax1.twinx()                   # Second y-axis on the right
ax2.plot(x, evr_cumul[:20] * 100,
         color='#E63946', marker='o',
         markersize=5, linewidth=2,
         label='Cumulative EVR')

ax2.set_ylabel('Cumulative Explained Variance (%)', color='#E63946', fontsize=12)
ax2.tick_params(axis='y', labelcolor='#E63946')
ax2.set_ylim(0, 105)

# Mark the 80%, 90%, 95% thresholds
for threshold, n, ls in [(80, n_80, '--'), (90, n_90, '-.'), (95, n_95, ':')]:
    ax2.axhline(y=threshold, color='gray', linestyle=ls, linewidth=1, alpha=0.7)
    ax2.axvline(x=n, color='gray', linestyle=ls, linewidth=1, alpha=0.7)
    ax2.text(n + 0.2, threshold + 1, f'{threshold}% @ PC{n}',
             fontsize=9, color='gray')

ax1.set_title('Scree Plot — Explained Variance of Principal Components\n'
              'Gas Sensor Array Drift Dataset (16 sensors, 128 features)',
              fontsize=12, fontweight='bold')

# Combined legend
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2,
           loc='center right', frameon=False)

plt.tight_layout()
plt.savefig(FIG_DIR + 'scree_plot.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Saved: scree_plot.png")

In [ ]:
from matplotlib.patches import Ellipse
import matplotlib.transforms as transforms

def confidence_ellipse(x, y, ax, n_std=2.0, **kwargs):
    """
    Draw a confidence ellipse for 2D data points (x, y).

    n_std=2.0 corresponds approximately to 95% confidence
    for a 2D Gaussian distribution.

    HOW IT WORKS:
    1. Compute the 2x2 covariance matrix of (x, y)
    2. Eigendecompose it → get axes directions and lengths
    3. Draw an ellipse scaled by n_std standard deviations

    This is a standard technique in chemometrics papers.
    """
    cov    = np.cov(x, y)                              # 2x2 covariance matrix
    pearsr = cov[0,1] / np.sqrt(cov[0,0] * cov[1,1])  # Pearson r between PC1,PC2

    # Ellipse angle from covariance structure
    ell_radius_x = np.sqrt(1 + pearsr)
    ell_radius_y = np.sqrt(1 - pearsr)

    ellipse = Ellipse((0, 0),
                      width  = ell_radius_x * 2,
                      height = ell_radius_y * 2,
                      **kwargs)

    # Scale ellipse to data spread
    scale_x = np.sqrt(cov[0,0]) * n_std
    scale_y = np.sqrt(cov[1,1]) * n_std
    mean_x, mean_y = np.mean(x), np.mean(y)

    transform = (transforms.Affine2D()
                 .rotate_deg(45)
                 .scale(scale_x, scale_y)
                 .translate(mean_x, mean_y))

    ellipse.set_transform(transform + ax.transData)
    return ax.add_patch(ellipse)


# ── Project data to 2D using sklearn PCA ──────────────────────
pca_2d = PCA(n_components=2)
Z_2d   = pca_2d.fit_transform(X_normalized)
# Z_2d shape: (13910, 2)
# Z_2d[:,0] = PC1 coordinate for each sample
# Z_2d[:,1] = PC2 coordinate for each sample

pc1_var = pca_2d.explained_variance_ratio_[0] * 100
pc2_var = pca_2d.explained_variance_ratio_[1] * 100

print(f"PC1 explains: {pc1_var:.2f}%")
print(f"PC2 explains: {pc2_var:.2f}%")
print(f"Together    : {pc1_var+pc2_var:.2f}%")

# ── Plot ───────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 7))

for gas_num, gas_name in gas_names.items():
    mask  = (y == gas_num)
    pc1   = Z_2d[mask, 0]
    pc2   = Z_2d[mask, 1]
    color = gas_colors[gas_name]
    count = mask.sum()

    # Scatter points
    ax.scatter(pc1, pc2,
               c=color, alpha=0.25,        # Transparent so overlaps show
               s=8,                         # Small dot size
               rasterized=True)            # Faster rendering for many points

    # Confidence ellipse
    confidence_ellipse(pc1, pc2, ax,
                       n_std=2.0,
                       edgecolor=color,
                       facecolor=color,
                       alpha=0.15,
                       linewidth=2)

    # Solid ellipse border
    confidence_ellipse(pc1, pc2, ax,
                       n_std=2.0,
                       edgecolor=color,
                       facecolor='none',
                       linewidth=2,
                       label=f'{gas_name} (n={count})')

    # Label the cluster centre
    ax.text(np.mean(pc1), np.mean(pc2), gas_name,
            fontsize=9, fontweight='bold',
            color=color,
            ha='center', va='center',
            bbox=dict(boxstyle='round,pad=0.2',
                      facecolor='white',
                      edgecolor=color,
                      alpha=0.8))

ax.set_xlabel(f'PC1 ({pc1_var:.1f}% variance explained)', fontsize=12)
ax.set_ylabel(f'PC2 ({pc2_var:.1f}% variance explained)', fontsize=12)
ax.set_title('Electronic Nose Fingerprinting via PCA\n'
             'Gas Sensor Array Drift Dataset — All Batches (n=13,910)',
             fontsize=13, fontweight='bold')
ax.legend(loc='upper right', frameon=False, fontsize=9)
ax.axhline(0, color='gray', linewidth=0.5, alpha=0.5)
ax.axvline(0, color='gray', linewidth=0.5, alpha=0.5)

plt.tight_layout()
plt.savefig(FIG_DIR + 'pca_2d_scatter.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Saved: pca_2d_scatter.png")

In [ ]:

batch_cmap = plt.cm.plasma
batch_colors = [batch_cmap(i/9) for i in range(10)]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))


ax = axes[0]
for gas_num, gas_name in gas_names.items():
    mask = (y == gas_num)
    ax.scatter(Z_2d[mask, 0], Z_2d[mask, 1],
               c=gas_colors[gas_name],
               alpha=0.3, s=6, rasterized=True,
               label=gas_name)

ax.set_xlabel(f'PC1 ({pc1_var:.1f}%)', fontsize=11)
ax.set_ylabel(f'PC2 ({pc2_var:.1f}%)', fontsize=11)
ax.set_title('Colored by Gas Identity', fontsize=12, fontweight='bold')
ax.legend(frameon=False, fontsize=8, markerscale=2)
ax.axhline(0, color='gray', linewidth=0.5, alpha=0.4)
ax.axvline(0, color='gray', linewidth=0.5, alpha=0.4)


ax = axes[1]
for batch_num in range(1, 11):
    mask = (batch_ids == batch_num)
    ax.scatter(Z_2d[mask, 0], Z_2d[mask, 1],
               c=[batch_colors[batch_num-1]],   # Color = time
               alpha=0.4, s=6, rasterized=True,
               label=f'Batch {batch_num}')

ax.set_xlabel(f'PC1 ({pc1_var:.1f}%)', fontsize=11)
ax.set_ylabel(f'PC2 ({pc2_var:.1f}%)', fontsize=11)
ax.set_title('Colored by Batch (Time Period)', fontsize=12, fontweight='bold')
ax.legend(frameon=False, fontsize=8, markerscale=2,
          loc='upper right', ncol=2)
ax.axhline(0, color='gray', linewidth=0.5, alpha=0.4)
ax.axvline(0, color='gray', linewidth=0.5, alpha=0.4)

fig.suptitle('Sensor Drift Revealed by PCA — Same Gases, Different Time Periods\n'
             'Gas Sensor Array Drift Dataset (n=13,910, 10 batches over 36 months)',
             fontsize=12, fontweight='bold', y=1.01)

plt.tight_layout()
plt.savefig(FIG_DIR + 'drift_visualization.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Saved: drift_visualization.png")

In [ ]:

print("═" * 60)
print("CLUSTER CENTROID SHIFT ACROSS BATCHES (in PC space)")
print("═" * 60)
print(f"\n{'Gas':<15} {'Batch':>6} {'PC1 centroid':>14} {'PC2 centroid':>14}")
print("─" * 55)


centroid_data = {}

for gas_num, gas_name in gas_names.items():
    centroids = []
    for batch_num in range(1, 11):

        mask = (y == gas_num) & (batch_ids == batch_num)

        if mask.sum() < 5:
            continue

        pc1_mean = Z_2d[mask, 0].mean()
        pc2_mean = Z_2d[mask, 1].mean()
        centroids.append((batch_num, pc1_mean, pc2_mean))

        print(f"{gas_name:<15} {batch_num:>6} {pc1_mean:>14.4f} {pc2_mean:>14.4f}")

    centroid_data[gas_name] = centroids
    print()


print("═" * 60)
print("TOTAL DRIFT (Euclidean distance: first → last centroid)")
print("═" * 60)

for gas_name, centroids in centroid_data.items():
    if len(centroids) < 2:
        continue
    first = np.array(centroids[0][1:])    # (PC1, PC2) of first batch
    last  = np.array(centroids[-1][1:])   # (PC1, PC2) of last batch
    drift = np.linalg.norm(last - first)  # Euclidean distance
    # np.linalg.norm computes: sqrt((Δpc1)² + (Δpc2)²)
    print(f"  {gas_name:<15}: {drift:.4f} PC-space units")

In [ ]:

fig, ax = plt.subplots(figsize=(10, 7))


ax.scatter(Z_2d[:, 0], Z_2d[:, 1],
           c='lightgray', alpha=0.1, s=4,
           rasterized=True, zorder=1)

for gas_name, centroids in centroid_data.items():
    if len(centroids) < 2:
        continue

    color      = gas_colors[gas_name]
    batch_nums = [c[0] for c in centroids]
    pc1s       = [c[1] for c in centroids]
    pc2s       = [c[2] for c in centroids]

    # Draw trajectory line
    ax.plot(pc1s, pc2s,
            color=color, linewidth=1.5,
            alpha=0.7, zorder=2)

    # Draw each centroid as a circle, sized and colored by batch
    for i, (bn, p1, p2) in enumerate(zip(batch_nums, pc1s, pc2s)):
        size = 60 + i * 20          # Later batches = larger circle
        ax.scatter(p1, p2,
                   c=[color], s=size,
                   edgecolors='white', linewidth=0.8,
                   zorder=3, alpha=0.9)
        ax.text(p1, p2, str(bn),
                fontsize=6, ha='center', va='center',
                color='white', fontweight='bold', zorder=4)

    # Label the gas name at its first centroid
    ax.text(pc1s[0] - 0.3, pc2s[0],
            gas_name, fontsize=8,
            color=color, fontweight='bold', zorder=5)

ax.set_xlabel(f'PC1 ({pc1_var:.1f}% variance explained)', fontsize=12)
ax.set_ylabel(f'PC2 ({pc2_var:.1f}% variance explained)', fontsize=12)
ax.set_title('Sensor Drift Trajectories in PC Space\n'
             'Numbers indicate batch (1=Month 0, 10=Month 36)',
             fontsize=12, fontweight='bold')
ax.axhline(0, color='gray', linewidth=0.5, alpha=0.4)
ax.axvline(0, color='gray', linewidth=0.5, alpha=0.4)

# Add time arrow annotation
ax.annotate('Time →\n(36 months)', xy=(0.02, 0.05),
            xycoords='axes fraction',
            fontsize=9, color='gray',
            bbox=dict(boxstyle='round', facecolor='white',
                      edgecolor='gray', alpha=0.8))

plt.tight_layout()
plt.savefig(FIG_DIR + 'drift_trajectories.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Saved: drift_trajectories.png")

In [ ]:

loadings = pca_2d.components_         # Shape: (2, 128)
pc1_loadings = loadings[0]            # 128 values for PC1
pc2_loadings = loadings[1]            # 128 values for PC2

print(f"Loadings shape: {loadings.shape}")
print(f"PC1 loadings — min: {pc1_loadings.min():.4f}, max: {pc1_loadings.max():.4f}")
print(f"PC2 loadings — min: {pc2_loadings.min():.4f}, max: {pc2_loadings.max():.4f}")


top5_pc1 = np.argsort(np.abs(pc1_loadings))[::-1][:5]
top5_pc2 = np.argsort(np.abs(pc2_loadings))[::-1][:5]

print("\n── Top 5 features driving PC1 ──")
for rank, idx in enumerate(top5_pc1, 1):
    sensor_num = (idx // 8) + 1      # Each sensor has 8 features
    feat_type  = idx % 8             # Which of the 8 features
    print(f"  Rank {rank}: Feature {idx+1:3d} "
          f"(Sensor {sensor_num}, type {feat_type}) "
          f"— loading: {pc1_loadings[idx]:+.4f}")

print("\n── Top 5 features driving PC2 ──")
for rank, idx in enumerate(top5_pc2, 1):
    sensor_num = (idx // 8) + 1
    feat_type  = idx % 8
    print(f"  Rank {rank}: Feature {idx+1:3d} "
          f"(Sensor {sensor_num}, type {feat_type}) "
          f"— loading: {pc2_loadings[idx]:+.4f}")

In [ ]:

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

feature_indices = np.arange(1, 129)   # 1 to 128


ax = axes[0]
colors_pc1 = ['#E63946' if v > 0 else '#2196F3' for v in pc1_loadings]
ax.bar(feature_indices, pc1_loadings,
       color=colors_pc1, alpha=0.8, width=0.8)
ax.axhline(0, color='black', linewidth=0.8)


for idx in top5_pc1:
    ax.bar(idx+1, pc1_loadings[idx],
           color='black', alpha=0.9, width=0.8)

ax.set_ylabel('PC1 Loading', fontsize=11)
ax.set_title('PC1 Loadings — Features Driving Primary Axis of Variation\n'
             '(Black bars = top 5 most influential features)',
             fontsize=11, fontweight='bold')
ax.set_xlim(0, 129)

# Add sensor boundary lines
for s in range(1, 17):
    ax.axvline(x=s*8 + 0.5, color='gray',
               linewidth=0.4, linestyle='--', alpha=0.5)

# Label sensor groups at top
for s in range(16):
    ax.text(s*8 + 4.5, ax.get_ylim()[1] * 0.85,
            f'S{s+1}', fontsize=6, ha='center',
            color='gray')


ax = axes[1]
colors_pc2 = ['#E63946' if v > 0 else '#2196F3' for v in pc2_loadings]
ax.bar(feature_indices, pc2_loadings,
       color=colors_pc2, alpha=0.8, width=0.8)
ax.axhline(0, color='black', linewidth=0.8)

for idx in top5_pc2:
    ax.bar(idx+1, pc2_loadings[idx],
           color='black', alpha=0.9, width=0.8)

ax.set_xlabel('Feature Index (8 features per sensor × 16 sensors)', fontsize=11)
ax.set_ylabel('PC2 Loading', fontsize=11)
ax.set_title('PC2 Loadings — Features Driving Secondary Axis of Variation',
             fontsize=11, fontweight='bold')
ax.set_xlim(0, 129)

for s in range(1, 17):
    ax.axvline(x=s*8 + 0.5, color='gray',
               linewidth=0.4, linestyle='--', alpha=0.5)

for s in range(16):
    ax.text(s*8 + 4.5, ax.get_ylim()[1] * 0.85,
            f'S{s+1}', fontsize=6, ha='center',
            color='gray')

# Shared legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#E63946', alpha=0.8, label='Positive loading'),
    Patch(facecolor='#2196F3', alpha=0.8, label='Negative loading'),
    Patch(facecolor='black',   alpha=0.9, label='Top 5 features'),
]
fig.legend(handles=legend_elements,
           loc='lower center', ncol=3,
           frameon=False, fontsize=10,
           bbox_to_anchor=(0.5, -0.02))

fig.suptitle('PCA Loading Vectors — Sensor Contribution to Principal Components\n'
             'Gas Sensor Array Drift Dataset (128 features, 16 sensors)',
             fontsize=12, fontweight='bold', y=1.01)

plt.tight_layout()
plt.savefig(FIG_DIR + 'loading_plot.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Saved: loading_plot.png")

Sensor 8 contributes most strongly to PC1, appearing twice in the top 5 loading rankings. This indicates that Sensor 8 captures the primary axis of chemical variation across the six gas classes. In a hardware-constrained deployment, Sensor 8 would be the last sensor to remove from the array.